# Fine-tune Whisper small with LoRA on a free GPU (Steps 6.2b, 6.3 and 6.4)

This notebook runs the project's own fine-tuning pipeline end to end on a **free Google Colab or Kaggle GPU** evaluates it against the base model on held-out lectures, and saves the adapter and the report.

| Step | What happens | Command |
|---|---|---|
| 1 | Clone the repo and install the `[finetune]` extra (torch, transformers, peft) | `pip install -e ".[finetune]"` |
| 2 | Download the MIT OpenCourseWare eval lectures and their official captions | `notetaker evalset` |
| 3 | Cut them into labeled clips of at most 30 s, split train / val / test by lecture | `notetaker finetune-data` |
| 4 | Pick a batch size that fits this GPU and write the training config | `notetaker.runtime.notebook_config` |
| 5 | Train LoRA adapters on Whisper small, keep the best by validation loss | `notetaker finetune` |
| 6 | Plot the loss curves | `finetune.json` |
| 7 | Quick look: base vs fine-tuned on a few validation clips | `load_finetuned`, `transcribe_clips` |
| 8 | Evaluate base vs fine-tuned on the held-out test lectures: WER, term recall, 95% interval | `notetaker finetune-eval` |
| 9 | Merge the adapter into Whisper and export an int8 CTranslate2 model that faster-whisper loads | `notetaker export-ct2` |
| 10 | Zip and save the adapter (a few MB) and the report | |

**Cost:** zero. Whisper small and all libraries are free and open; no API key or Hugging Face token is needed. **Time:** about 10 to 20 minutes for downloads and clips, then roughly 30 to 60 minutes for 1000 training steps on a T4.

**License:** the OCW lectures are CC BY-NC-SA 4.0. They are downloaded into this session only and must not be re-uploaded; the saved adapter is a small set of weight corrections, not the audio.

**First run tip:** leave `SMOKE_TEST = True` in the settings cell. It trains for 20 steps (about a minute) so any problem shows up early. Then set it to `False` and run steps 4 to 10 again for the real run.

## 0. Turn on a GPU

- **Colab:** Runtime > Change runtime type > **T4 GPU** > Save.
- **Kaggle:** in the right panel, Session options > Accelerator > **GPU T4 x2** (one GPU is used) and turn **Internet on** (Kaggle asks for phone verification once).

The cell below should print a table with a Tesla T4 (or P100). If it says `command not found`, there is no GPU attached.

In [ ]:
!nvidia-smi

## 1. Settings, clone and install

Edit these values, then run the cells in order. `SMOKE_TEST = True` writes to a separate `-smoke` folder, so it never overwrites a real run.

In [ ]:
REPO_URL = "https://github.com/Aksh-99/ai-note-taker.git"
BRANCH = "main"

SMOKE_TEST = True  # 20-step check first; set False for the real run
MAX_STEPS = 1000  # optimizer steps for the real run
USE_DRIVE = False  # Colab only: also copy the result to Google Drive

# Which eval lectures to use (ids from eval/ocw_lectures.json). None = all.
LECTURES = None

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Where to put the code. Colab's scratch disk is /content; on Kaggle only
# /kaggle/working is saved, so the big checkout + audio go to /kaggle/temp.
# (notetaker.runtime.detect_platform does this same check; it cannot be
# imported yet because the package is not installed.)
if "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ:
    REPO_DIR = Path("/content/ai-note-taker")
elif "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    REPO_DIR = Path("/kaggle/temp/ai-note-taker")
elif Path("pyproject.toml").exists() and Path("src/notetaker").is_dir():
    REPO_DIR = Path.cwd()  # running locally inside the repo already
else:
    REPO_DIR = Path.cwd() / "ai-note-taker"

if not (REPO_DIR / "pyproject.toml").exists():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
print("Repo:", REPO_DIR)

In [ ]:
# Colab and Kaggle already ship a CUDA build of torch, so pip keeps it and
# only adds transformers, peft and the project itself.
!pip install -q -e "{REPO_DIR}[finetune]"

In [ ]:
# `pip install -e` registers the package for NEW Python processes; this
# already-running notebook kernel does not see it until restarted. Adding
# src/ to sys.path makes `import notetaker` work right away. The
# `!notetaker ...` shell commands below are new processes, so they are fine.
sys.path.insert(0, str(REPO_DIR / "src"))

from notetaker.runtime import default_paths, detect_platform

PLATFORM = detect_platform()
drive_mounted = False
if PLATFORM == "colab" and USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    drive_mounted = True

paths = default_paths(PLATFORM, drive_mounted=drive_mounted)
DATA_DIR = paths.data
CLIPS_DIR = DATA_DIR / "finetune"
OUT_DIR = paths.output
if SMOKE_TEST:
    OUT_DIR = OUT_DIR.with_name(OUT_DIR.name + "-smoke")

# Every notetaker command reads its data folder from this variable, and
# `!` commands inherit the notebook's environment variables.
os.environ["NOTETAKER_DATA_DIR"] = str(DATA_DIR)
# Run from the repo so default paths like eval/ocw_lectures.json resolve.
os.chdir(REPO_DIR)
ONLY = ("--only " + " ".join(LECTURES)) if LECTURES else ""

print(f"Platform: {PLATFORM}")
print(f"Data:     {DATA_DIR}")
print(f"Output:   {OUT_DIR}")
print(f"Save to:  {paths.persist}")

## 2. Download the eval lectures

For each lecture: the official caption file (the reference transcript) and the audio track, streamed by ffmpeg straight from the video URL and saved as 16 kHz mono FLAC. Files that already exist are skipped, so re-running is cheap.

In [ ]:
!notetaker evalset -o {DATA_DIR}/raw/ocw {ONLY}

## 3. Build the fine-tuning clips

First a dry run (captions only) to see how many clips and hours each split gets, then the real build. Splits are by **whole lecture**, so no lecture, voice or sentence appears in both train and test (see Step 6.1 in the DEVLOG). `--force` only replaces folders this command made.

In [ ]:
!notetaker finetune-data --eval-dir {DATA_DIR}/raw/ocw -o {CLIPS_DIR} --dry-run

In [ ]:
!notetaker finetune-data --eval-dir {DATA_DIR}/raw/ocw -o {CLIPS_DIR} --force

## 4. Training config for this GPU

Batch 8 fits a 16 GB T4 in fp16. On a smaller card the batch is halved and gradient accumulation doubled, so every optimizer step still averages 16 clips and the training behaves the same, only slower. The config is saved as JSON in the output folder so the run can be repeated exactly.

In [ ]:
import torch

from notetaker.runtime import notebook_config, write_config

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    gpu_gb = props.total_memory / 1e9
    print(f"GPU: {props.name}, {gpu_gb:.1f} GB")
else:
    gpu_gb = None
    print("No GPU found: only the smoke test is realistic on CPU (see step 0).")

config = notebook_config(gpu_gb, max_steps=MAX_STEPS, smoke_test=SMOKE_TEST)
# Saved inside the output folder, so it travels with the adapter zip.
CONFIG_PATH = write_config(config, OUT_DIR / "train_config.json")
print(
    f"batch {config.batch_size} x accumulation {config.grad_accum_steps}, "
    f"{config.max_steps} steps, fp16={config.fp16}\nConfig: {CONFIG_PATH}"
)

## 5. Fine-tune

The dry run prints the plan (clips, hours, steps per epoch, epochs) without loading a model. The real run downloads Whisper small (about 1 GB, free, no account) on first use, then logs the training loss and, every `eval_every` steps, the validation loss. The adapter is saved each time the validation loss improves.

In [ ]:
!notetaker finetune --data-dir {CLIPS_DIR} --config {CONFIG_PATH} --dry-run

In [ ]:
!notetaker finetune --data-dir {CLIPS_DIR} --config {CONFIG_PATH} -o {OUT_DIR} --force

## 6. Loss curves

Training loss should fall steadily. Validation loss usually falls and then flattens or rises: the point where it is lowest is the saved adapter. A large gap between the two curves means overfitting (memorising the training lectures), which is the main risk with only a few hours of audio.

In [ ]:
import json

import matplotlib.pyplot as plt

summary = json.loads((OUT_DIR / "finetune.json").read_text())
history = summary["history"]
train = [(row["step"], row["train_loss"]) for row in history]
val = [(row["step"], row["val_loss"]) for row in history if row["val_loss"] is not None]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(*zip(*train, strict=True), label="train", alpha=0.7)
if val:
    ax.plot(*zip(*val, strict=True), marker="o", label="validation")
ax.axvline(summary["best_step"], color="gray", linestyle="--", label="best (saved)")
ax.set_xlabel("optimizer step")
ax.set_ylabel("cross-entropy loss")
ax.set_title("LoRA fine-tuning of Whisper small")
ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "loss_curve.png", dpi=150)
plt.show()

params = summary["parameters"]
print(
    f"Trainable parameters: {params['trainable']:,} of {params['total']:,} "
    f"({100 * params['trainable'] / params['total']:.2f}%)"
)
print(f"Best validation loss {summary['best_val_loss']} at step {summary['best_step']}")

## 7. Quick look: base vs fine-tuned

Transcribes a few **validation** clips twice with the same model object: once with the adapters switched off (`model.disable_adapter()`, which is exactly base Whisper small) and once with them on. This is a sanity check, not the evaluation: the proper comparison on the held-out **test** lectures, with WER and technical-term recall, is the next section.

In [ ]:
from notetaker.finetune import load_clips, load_finetuned, transcribe_clips
from notetaker.wer import ErrorCounts, score_texts

N_CLIPS = 6
clips = load_clips(CLIPS_DIR, "val")[:N_CLIPS]
model, feature_extractor, tokenizer, _ = load_finetuned(OUT_DIR)

with model.disable_adapter():
    base_texts = transcribe_clips(model, feature_extractor, tokenizer, clips)
tuned_texts = transcribe_clips(model, feature_extractor, tokenizer, clips)

# Pool word errors over all clips (sum the counts, then divide), the same
# corpus-level WER the eval reports use, instead of averaging per-clip WERs.
pooled = {"base": ErrorCounts(), "fine-tuned": ErrorCounts()}
for clip, base, tuned in zip(clips, base_texts, tuned_texts, strict=True):
    pooled["base"] += score_texts(clip.text, base).words
    pooled["fine-tuned"] += score_texts(clip.text, tuned).words
    print(f"REF:   {clip.text}\nBASE:  {base}\nTUNED: {tuned}\n")
for name, counts in pooled.items():
    print(f"{name:>10} WER on these {len(clips)} clips: {100 * counts.rate:.1f}%")

## 8. Evaluate: base vs fine-tuned on the test lectures

The real evaluation (Step 6.3). Every clip of the **test** split (lectures never used for training or for picking the best step) is transcribed twice by the same model object, adapter off and on. `notetaker finetune-eval` then reports pooled WER, CER and technical-term recall for both, the change per lecture, the terms that were fixed or broken, a few example clips, and a **paired bootstrap 95% interval** for the WER change: if the whole interval is below zero, the improvement is unlikely to be luck in which clips ended up in the test set.

The transcripts are saved next to the adapter as `eval_transcripts.jsonl`, so the report can be rebuilt later on any laptop without a GPU: `notetaker finetune-eval --from-transcripts <unzipped folder>/eval_transcripts.jsonl --model-dir <unzipped folder>`.

In [ ]:
REPORT_DIR = OUT_DIR / "report"
!notetaker finetune-eval --model-dir {OUT_DIR} --data-dir {CLIPS_DIR} -o {REPORT_DIR}

In [ ]:
from IPython.display import Markdown, display

display(Markdown((REPORT_DIR / "finetune.md").read_text()))

## 9. Export for fast inference (int8 CTranslate2)

The rest of the project runs Whisper through faster-whisper (the CTranslate2 engine), not PyTorch. `notetaker export-ct2` folds the LoRA correction into the weights once (W + (alpha / r) B A, so inference pays nothing extra for the adapter), converts the merged model to CTranslate2 with int8 weights (about 4x smaller than float32), and checks that the folder loads in faster-whisper. It runs on the CPU in about a minute.

The export goes to a folder next to `OUT_DIR` and is **not** added to the zip below (it is a few hundred MB). The same command rebuilds it on your laptop from the small adapter: `notetaker export-ct2 --model-dir models/<run folder>`.

In [ ]:
CT2_DIR = OUT_DIR.with_name(OUT_DIR.name + "-ct2-int8")
!notetaker export-ct2 --model-dir {OUT_DIR} --out {CT2_DIR} --force

## 10. Save the adapter and the report

Zips `adapter/` (the LoRA weights, a few MB), `finetune.json` (settings, data plan and loss history), `train_config.json`, the loss plot, `eval_transcripts.jsonl` and `report/` (the evaluation from step 8). On Kaggle the zip lands in `/kaggle/working` and appears under the notebook's **Output** tab; on Colab it is downloaded to your computer (or copied to Google Drive if `USE_DRIVE = True`). Unzip it into the repo's `models/` folder (gitignored), and copy `report/finetune.md` and `report/finetune.json` into the repo's `reports/` folder.

In [ ]:
import shutil

paths.persist.mkdir(parents=True, exist_ok=True)
base_name = str(paths.persist / OUT_DIR.name)
archive = shutil.make_archive(base_name, "zip", root_dir=OUT_DIR)
print("Saved", archive)

if PLATFORM == "colab" and not drive_mounted:
    from google.colab import files

    files.download(archive)